# Inventory Demand Prediction

In [1]:
import numpy as np
import pandas as pd

### Loading The DataSet

In [2]:
df = pd.read_csv("../data/raw/train.csv")

df

,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10
...,...,...,...,...
912995,2017-12-27,10,50,63
912996,2017-12-28,10,50,59
912997,2017-12-29,10,50,74
912998,2017-12-30,10,50,62


### Dataset Exploring

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 913000 entries, 0 to 912999
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   date    913000 non-null  str  
 1   store   913000 non-null  int64
 2   item    913000 non-null  int64
 3   sales   913000 non-null  int64
dtypes: int64(3), str(1)
memory usage: 27.9 MB


In [4]:
df.shape

(913000, 4)

In [5]:
df.describe()

,store,item,sales
count,913000.000000,913000.000000,913000.000000
mean,5.500000,25.500000,52.250287
std,2.872283,14.430878,28.801144
min,1.000000,1.000000,0.000000
25%,3.000000,13.000000,30.000000
50%,5.500000,25.500000,47.000000
75%,8.000000,38.000000,70.000000
max,10.000000,50.000000,231.000000


NO Missing Value are there so now it's time to do feature enginnering


## Feature Engineering

In [6]:
new_df = df.copy()

new_df['date'] = pd.to_datetime(new_df['date'])

new_df['year'] = new_df['date'].dt.year
new_df['month'] = new_df['date'].dt.month
new_df['day'] = new_df['date'].dt.day
new_df['dayofweek'] = new_df['date'].dt.dayofweek

new_df['is_weekend'] = (new_df['dayofweek'] >= 5).astype(int)

new_df

,date,store,item,sales,year,month,day,dayofweek,is_weekend
0,2013-01-01,1,1,13,2013,1,1,1,0
1,2013-01-02,1,1,11,2013,1,2,2,0
2,2013-01-03,1,1,14,2013,1,3,3,0
3,2013-01-04,1,1,13,2013,1,4,4,0
4,2013-01-05,1,1,10,2013,1,5,5,1
...,...,...,...,...,...,...,...,...,...
912995,2017-12-27,10,50,63,2017,12,27,2,0
912996,2017-12-28,10,50,59,2017,12,28,3,0
912997,2017-12-29,10,50,74,2017,12,29,4,0
912998,2017-12-30,10,50,62,2017,12,30,5,1


In [15]:
group_sales = new_df.groupby(["store","item"])['sales']

new_df['lag_1'] = group_sales.shift(1)
new_df['lag_7'] = group_sales.shift(7)
new_df['lag_30'] = group_sales.shift(30)

new_df['rolling_7_mean'] = ( group_sales.shift(1).rolling(window=7).mean())
new_df['rolling_30_mean'] = ( group_sales.shift(1).rolling(window=30).mean())
new_df['rolling_7_std'] = ( group_sales.shift(1).rolling(window=7).std())
new_df['rolling_30_std'] = ( group_sales.shift(1).rolling(window=30).std())

new_df = new_df.fillna(-1)

new_df.to_csv('filename.csv', index=False)


## Data Splitting Process

In [8]:
train_df = df[df['date'] < '2017-01-01'].copy()
test_df = df[df['date'] >= '2017-01-01'].copy()

# Model Training



In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error ,  mean_absolute_percentage_error 
import xgboost as xgb
import lightgbm as lgb

In [10]:
features = [col for col in train_df.columns if col not in ['date', 'sales']]
    

X_train = train_df[features]
y_train = train_df['sales']
    

X_test = test_df[features]
y_test = test_df['sales']

In [11]:
def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [12]:
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
    
lgb_model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
lgb_model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004580 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 62
[LightGBM] [Info] Number of data points in the train set: 730500, number of used features: 2
[LightGBM] [Info] Start training from score 50.610229


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [ ]:
xgb_preds = xgb_model.predict(X_test)
lgb_preds = lgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_preds)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_mape = calculate_mape(y_test, xgb_preds)

print(f"\nXGBoost")
print(f"MAE: {xgb_mae:.4f} | RMSE: {xgb_rmse:.4f} | MAPE: {xgb_mape:.2f}%")

lgb_mae = mean_absolute_error(y_test, lgb_preds)
lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_preds))
lgb_mape = calculate_mape(y_test, lgb_preds)

print(f"\nLightGBM")
print(f"MAE: {lgb_mae:.4f} | RMSE: {lgb_rmse:.4f} | MAPE: {lgb_mape:.2f}%")


--- XGBoost Test Metrics ---
MAE: 13.8548 | RMSE: 18.9447 | MAPE: 23.77%

--- LightGBM Test Metrics ---
MAE: 13.7791 | RMSE: 18.7028 | MAPE: 23.68%


In [14]:
import itertools

xgb_param_grid = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 200],
}

X_train = train_df[features]
y_train = train_df["sales"]

X_test = test_df[features]
y_test = test_df["sales"]

scorecard = []

keys, values = zip(*xgb_param_grid.items())
param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]


for params in param_combinations:
    print(f"Testing profile: {params}", end=" ")

    model = xgb.XGBRegressor(
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        n_estimators=params["n_estimators"],
        early_stopping_rounds=10,  # Stops training if 2017 metrics stall for 10 rounds
        random_state=42,
        n_jobs=-1,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=False,  # Stops individual tree epoch outputs from spamming notebook
    )

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    mape = mean_absolute_percentage_error(y_test, preds)

    print(f"Finished -> MAPE: {mape:.4%}" , end=" ")

    scorecard.append(
        {
            "max_depth": params["max_depth"],
            "learning_rate": params["learning_rate"],
            "n_estimators": model.best_iteration + 1, 
            "MAE": mae,
            "MAPE": mape,
        }
    )

results_df = pd.DataFrame(scorecard)

best_run = results_df.loc[results_df["MAPE"].idxmin()]

print("\n" + "=" * 60)
print("🏆 EXPERIMENT COMPLETE: WINNING PARAMETERS FOUND")
print("=" * 60)
print(f"Best Validation MAE:   {best_run['MAE']:.4f}")
print(f"Best Validation MAPE:  {best_run['MAPE']:.4%}")
print("=" * 60)

Testing profile: {'max_depth': 3, 'learning_rate': 0.01, 'n_estimators': 100} Finished -> MAPE: 46.4753% Testing profile: {'max_depth': 3, 'learning_rate': 0.01, 'n_estimators': 200} Finished -> MAPE: 41.6645% Testing profile: {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 100} Finished -> MAPE: 33.5273% Testing profile: {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 200} Finished -> MAPE: 27.3643% Testing profile: {'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 100} Finished -> MAPE: 27.3556% Testing profile: {'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 200} Finished -> MAPE: 24.7350% Testing profile: {'max_depth': 5, 'learning_rate': 0.01, 'n_estimators': 100} Finished -> MAPE: 42.6618% Testing profile: {'max_depth': 5, 'learning_rate': 0.01, 'n_estimators': 200} Finished -> MAPE: 36.8253% Testing profile: {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 100} 

Exception ignored on calling ctypes callback function: <bound method DataIter._next_wrapper of <xgboost.data.SingleBatchInternalIter object at 0x7416676ecef0>>
Traceback (most recent call last):
  File "/home/manav-tejani/Desktop/dev/envs/cancer-venv/lib/python3.12/site-packages/xgboost/core.py", line 662, in _next_wrapper
    return self._handle_exception(lambda: int(self.next(input_data)), 0)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/manav-tejani/Desktop/dev/envs/cancer-venv/lib/python3.12/site-packages/xgboost/core.py", line 575, in _handle_exception
    return fn()
           ^^^^
  File "/home/manav-tejani/Desktop/dev/envs/cancer-venv/lib/python3.12/site-packages/xgboost/core.py", line 662, in <lambda>
    return self._handle_exception(lambda: int(self.next(input_data)), 0)
                                              ^^^^^^^^^^^^^^^^^^^^^
  File "/home/manav-tejani/Desktop/dev/envs/cancer-venv/lib/python3.12/site-packages/xgboost/data

Finished -> MAPE: 27.3915% Testing profile: {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 200} 

XGBoostError: [17:19:04] /__w/xgboost/xgboost/src/data/iterative_dmatrix.cc:114: Check failed: accumulated_rows == this->info_.num_row_ (0 vs. 730500) : 
Stack trace:
  [bt] (0) /home/manav-tejani/Desktop/dev/envs/cancer-venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.so(+0x2c1a8c) [0x7416690c1a8c]
  [bt] (1) /home/manav-tejani/Desktop/dev/envs/cancer-venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.so(+0x67ccf2) [0x74166947ccf2]
  [bt] (2) /home/manav-tejani/Desktop/dev/envs/cancer-venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.so(+0x67e05d) [0x74166947e05d]
  [bt] (3) /home/manav-tejani/Desktop/dev/envs/cancer-venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.so(+0x5dadf1) [0x7416693dadf1]
  [bt] (4) /home/manav-tejani/Desktop/dev/envs/cancer-venv/lib/python3.12/site-packages/xgboost/lib/libxgboost.so(XGQuantileDMatrixCreateFromCallback+0x178) [0x741668fcf5a8]
  [bt] (5) /lib/x86_64-linux-gnu/libffi.so.8(+0x7b16) [0x7416ff1a3b16]
  [bt] (6) /lib/x86_

In [ ]:
xgb_preds = xgb_model.predict(X_test)
lgb_preds = lgb_model.predict(X_test)

best_mape = float("inf")
best_weight = 0.0
best_preds = None

print("🔬 Simulating Optimal Ensemble Weights...")
print("-" * 50)

for w in np.linspace(0, 1, 11):
    lgb_weight = 1.0 - w

    blended_preds = (w * xgb_preds) + (lgb_weight * lgb_preds)

    mae = mean_absolute_error(y_test, blended_preds)
    mape = np.mean(np.abs((y_test - blended_preds) / y_test)) * 100

    print(
        f"XGB Weight: {w:.1f} | LGB Weight: {lgb_weight:.1f} -> MAPE: {mape:.4f}% | MAE: {mae:.4f}"
    )

    if mape < best_mape:
        best_mape = mape
        best_weight = w
        best_preds = blended_preds


print("-" * 50)
print("🏆 OPTIMAL ENSEMBLE CONFIGURATION")
print(
    f"Best XGB Weight: {best_weight:.1f} | Best LGB Weight: {1.0-best_weight:.1f}"
)
print(f"Minified Validation MAPE: {best_mape:.4f}%")

🔬 Simulating Optimal Ensemble Weights...
--------------------------------------------------
XGB Weight: 0.0 | LGB Weight: 1.0 -> MAPE: 23.6760% | MAE: 13.7791
XGB Weight: 0.1 | LGB Weight: 0.9 -> MAPE: 23.6729% | MAE: 13.7822
XGB Weight: 0.2 | LGB Weight: 0.8 -> MAPE: 23.6722% | MAE: 13.7862
XGB Weight: 0.3 | LGB Weight: 0.7 -> MAPE: 23.6744% | MAE: 13.7913
XGB Weight: 0.4 | LGB Weight: 0.6 -> MAPE: 23.6802% | MAE: 13.7975
XGB Weight: 0.5 | LGB Weight: 0.5 -> MAPE: 23.6889% | MAE: 13.8047
XGB Weight: 0.6 | LGB Weight: 0.4 -> MAPE: 23.6998% | MAE: 13.8126
XGB Weight: 0.7 | LGB Weight: 0.3 -> MAPE: 23.7135% | MAE: 13.8216
XGB Weight: 0.8 | LGB Weight: 0.2 -> MAPE: 23.7302% | MAE: 13.8316
XGB Weight: 0.9 | LGB Weight: 0.1 -> MAPE: 23.7497% | MAE: 13.8425
XGB Weight: 1.0 | LGB Weight: 0.0 -> MAPE: 23.7733% | MAE: 13.8548
--------------------------------------------------
🏆 OPTIMAL ENSEMBLE CONFIGURATION
Best XGB Weight: 0.2 | Best LGB Weight: 0.8
Minified Validation MAPE: 23.6722%
